# 🧗 How Strong Am I Really? — Personal Climbing Assessment

This notebook gives you a **data-driven snapshot** of your current climbing ability
using models from peer-reviewed research. It answers:

| Question | Method |
|----------|--------|
| How strong are my fingers? | MVC-7 / body-weight ratio |
| What grade should I climb? | Composite & MaxToGrade models |
| Am I endurance- or strength-limited? | Critical Force / W′ analysis |
| What should I train next? | Automated weakness identification |

### What you need

* **Required** — Body weight, a 7-second max hang result (total load on a known edge).
* **Optional** — Time-to-failure tests at 80 %, 60 %, and 45 % MVC for endurance analysis.

> All models are referenced in the final section. Edge depth is normalised to the
> 20 mm reference standard automatically.

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

try:
    import climbing_science as cs
    from climbing_science import (
        mvc7_to_grade,
        grade_to_mvc7,
        rohmert_conversion,
        power_to_weight,
        correction_factor,
        normalize_to_reference,
        critical_force,
        cf_mvc_ratio,
        interpret_cf_ratio,
        classify_endurance,
        classify_level,
        identify_weakness,
        training_priority,
        convert,
        StrengthModel,
    )
    from climbing_science.frontends.notebook import (
        plot_strength_benchmark,
        plot_rohmert_curve,
    )
    print(f"✅ climbing-science v{cs.__version__} loaded")
except ImportError as exc:
    raise SystemExit(
        f"❌ Could not import climbing-science: {exc}\n"
        "   Install with: pip install climbing-science"
    )

## 📝 Your Data

**Edit the cell below** — this is the *only* cell you need to change.
All subsequent analysis derives from these values.

In [ ]:
# ── YOUR DATA (edit here) ─────────────────────────────────────────
BODYWEIGHT_KG = 72
HEIGHT_CM = 178
MVC7_KG = 105         # Total load (bodyweight + added weight) for a 7 s max hang
TEST_EDGE_MM = 20     # Edge depth used for the test (mm)
CURRENT_GRADE = "7a"  # Your current redpoint grade (French sport)

# Optional endurance data — set to None if you haven't tested
T80_S = 77            # Time-to-failure at 80 % MVC (seconds)
T60_S = 136           # Time-to-failure at 60 % MVC (seconds)
T45_S = 323           # Time-to-failure at 45 % MVC (seconds)
# ──────────────────────────────────────────────────────────────────

## Step 1 — Edge-Depth Correction

Finger strength measured on different edge depths is not directly comparable.
A shallower edge requires more force per unit of contact area, so we apply a
correction factor (Amca et al., 2012) to normalise everything to the **20 mm
reference edge**.

In [ ]:
if TEST_EDGE_MM != 20:
    cf = correction_factor(TEST_EDGE_MM)
    mvc7_normalised = normalize_to_reference(MVC7_KG, TEST_EDGE_MM)
    print(f"Edge correction factor ({TEST_EDGE_MM} mm → 20 mm): {cf:.2f}")
    print(f"Raw MVC-7: {MVC7_KG:.1f} kg  →  Normalised: {mvc7_normalised:.1f} kg")
else:
    mvc7_normalised = MVC7_KG
    print(f"Test edge is 20 mm — no correction needed (MVC-7 = {mvc7_normalised:.1f} kg)")

## Step 2 — Power-to-Weight Ratio

The MVC-7 / body-weight ratio is the single most predictive metric for climbing
grade (Giles et al., 2019; Lattice Training benchmarks). The ratio is expressed
as a percentage of body weight.

In [ ]:
ratio, level_label = power_to_weight(mvc7_normalised, BODYWEIGHT_KG)

print(f"MVC-7 (normalised):  {mvc7_normalised:.1f} kg")
print(f"Body weight:         {BODYWEIGHT_KG:.1f} kg")
print(f"Power-to-weight:     {ratio:.1f} % BW")
print(f"Classification:      {level_label}")

## Step 3 — Grade Prediction

Two independent models estimate what grade your finger strength *should*
support:

| Model | Description |
|-------|-------------|
| **COMPOSITE** | Regression across multiple studies — best for **route** climbing |
| **MAXTOGRADE** | Lattice-style lookup — better for **bouldering** single-move max |

In [ ]:
grade_composite = mvc7_to_grade(ratio, model=StrengthModel.COMPOSITE)
grade_boulder   = mvc7_to_grade(ratio, model=StrengthModel.MAXTOGRADE)

expected_ratio = grade_to_mvc7(CURRENT_GRADE)

print(f"Your ratio:                {ratio:.1f} % BW")
print(f"Predicted grade (route):   {grade_composite}  (COMPOSITE model)")
print(f"Predicted grade (boulder): {grade_boulder}  (MAXTOGRADE model)")
print()
print(f"Current redpoint grade:    {CURRENT_GRADE}")
print(f"Expected ratio for {CURRENT_GRADE}:  {expected_ratio:.1f} % BW")

diff = ratio - expected_ratio
if diff > 5:
    print(f"\n💡 You are {diff:.0f} pp STRONGER than needed for {CURRENT_GRADE} "
          "— technique or endurance may be limiting you.")
elif diff < -5:
    print(f"\n💪 You are climbing {-diff:.0f} pp ABOVE your strength level "
          "— great technique efficiency!")
else:
    print(f"\n✅ Strength and grade are well matched.")

## Step 4 — Strength Benchmark Plot

The plot below shows your MVC-7 / BW ratio against the climbing population.
The vertical line marks *your* value.

In [ ]:
plot_strength_benchmark(mvc_bw_ratio=ratio);

## Step 5 — Endurance Analysis (Critical Force / W′)

If you provided time-to-failure data at three intensities, we fit the
**critical force model** (Giles et al., 2019). This splits your capacity into:

* **CF** — the intensity you can sustain "indefinitely" (aerobic threshold)
* **W′** — your finite anaerobic reserve above CF

> *Skip this section if you set the T-values to `None` above.*

In [ ]:
HAS_ENDURANCE_DATA = all(v is not None for v in [T80_S, T60_S, T45_S])

if HAS_ENDURANCE_DATA:
    intensities = [80, 60, 45]
    durations = [T80_S, T60_S, T45_S]

    cf_pct, w_prime, r_squared = critical_force(intensities, durations)

    print(f"Critical Force (CF):  {cf_pct:.1f} % MVC")
    print(f"W′:                   {w_prime:.0f} % MVC·s")
    print(f"Model R²:             {r_squared:.3f}")

    # CF / MVC ratio
    cfr = cf_mvc_ratio(cf_pct, ratio)
    interpretation = interpret_cf_ratio(cfr)
    endurance_class = classify_endurance(cfr)

    print(f"\nCF / MVC ratio:       {cfr:.2f}")
    print(f"Category:             {interpretation['category']}")
    print(f"Classification:       {endurance_class}")
    print(f"\n📋 {interpretation['interpretation']}")
    print(f"🏋️ Recommendation:    {interpretation['priority']}")
else:
    cf_pct = None
    w_prime = None
    cfr = None
    print("ℹ️  No endurance data provided — skipping Critical Force analysis.")
    print("   To enable, fill in T80_S, T60_S, T45_S in the data cell above.")

## Step 6 — Climber Level Classification

A simple lookup based on your stated current redpoint grade.

In [ ]:
level = classify_level(CURRENT_GRADE)
print(f"Grade: {CURRENT_GRADE}  →  Level: {level}")

## Step 7 — Weakness Identification

Using your strength ratio and (optionally) endurance data, the library
identifies your **primary limiter**.

In [ ]:
weaknesses = identify_weakness(ratio, cfr)

if weaknesses:
    print("Identified weaknesses:")
    for i, w in enumerate(weaknesses, 1):
        print(f"  {i}. {w}")
else:
    print("✅ No clear weakness identified — well-rounded profile!")

## Step 8 — Training Priorities

Based on the identified weaknesses, here are the recommended training
priorities.

In [ ]:
priorities = training_priority(weaknesses)
print("🎯 Training priority:", priorities)

## 📊 Summary Report

In [ ]:
print("=" * 60)
print("  PERSONAL CLIMBING ASSESSMENT — SUMMARY")
print("=" * 60)
print()
print(f"  Athlete:          {BODYWEIGHT_KG} kg / {HEIGHT_CM} cm")
print(f"  Current grade:    {CURRENT_GRADE}  ({level})")
print(f"  MVC-7 (20 mm):    {mvc7_normalised:.1f} kg")
print(f"  Power-to-weight:  {ratio:.1f} % BW  ({level_label})")
print()
print(f"  Predicted grade")
print(f"    Route:          {grade_composite}  (COMPOSITE)")
print(f"    Boulder:        {grade_boulder}  (MAXTOGRADE)")
print()

if HAS_ENDURANCE_DATA:
    print(f"  Endurance")
    print(f"    Critical Force: {cf_pct:.1f} % MVC")
    print(f"    W′:             {w_prime:.0f} % MVC·s")
    print(f"    CF/MVC ratio:   {cfr:.2f}  ({interpretation['category']})")
    print()

if weaknesses:
    print(f"  Primary limiter:  {', '.join(weaknesses)}")
    print(f"  Training focus:   {priorities}")
else:
    print("  No clear limiter — balanced profile.")

print()
print("=" * 60)

## 📚 References

| Key | Citation |
|-----|----------|
| Rohmert 1960 | Rohmert, W. (1960). Ermittlung von Erholungspausen für statische Arbeit des Menschen. *Internationale Zeitschrift für angewandte Physiologie einschließlich Arbeitsphysiologie*, 18, 123–164. |
| Amca 2012 | Amca, A. M., et al. (2012). The effect of hold depth on maximal finger force in rock climbing. *Journal of Sports Sciences*, 30(7), 669–677. |
| Giles 2019 | Giles, D., et al. (2019). An all-out test to determine finger flexor critical force in rock climbers. *International Journal of Sports Physiology and Performance*, 14(7), 942–948. |
| Draper 2015 | Draper, N., et al. (2015). Comparative grading scales, statistical analyses, climber descriptors and ability groupings. *Sports Technology*, 8(1-2), 21–30. |
| Lattice | Lattice Training — Fingerboard assessment benchmarks (latticetraining.com). |

---
*Generated with [climbing-science](https://pypi.org/project/climbing-science/) v0.3.3*